# MOSS-TTS v1.5 — Kaggle T4×2 Production Demo / Demo Production trên Kaggle T4×2

## English

This notebook demonstrates the **original MOSS-TTS v1.5 BF16 safetensors checkpoint** on exactly **two NVIDIA Tesla T4 16 GB GPUs**. The deployment changes model placement and execution order to fit the qualified Kaggle hardware; it does **not** use GGUF, quantization, modified weights, or a modified architecture.

This is an independent engineering qualification project, not an official OpenMOSS release. Release qualification is backed by a pinned candidate T4×2 run plus explicit seven-sample human listening review. After v1.0.0 is published, this notebook is the public reproducibility/showcase entry point for a fresh Kaggle run.

## Tiếng Việt

Notebook này trình diễn **checkpoint safetensors BF16 nguyên bản của MOSS-TTS v1.5** trên đúng **hai GPU NVIDIA Tesla T4 16 GB**. Cách triển khai chỉ thay đổi vị trí đặt model và thứ tự thực thi để phù hợp phần cứng Kaggle đã qualify; **không** dùng GGUF, không quantization, không sửa weight và không thay đổi kiến trúc model.

Đây là dự án kiểm định kỹ thuật độc lập, không phải bản phát hành chính thức của OpenMOSS. Release qualification dựa trên candidate run T4×2 đã pin cùng human listening review đủ bảy mẫu; sau khi v1.0.0 được publish, notebook này là entry point để chạy fresh public reproducibility/showcase trên Kaggle.

## Before you run / Trước khi chạy

### English

Before running the notebook in Kaggle:

1. Import this notebook from `dangkhoa2016/MOSS-TTS-v1.5-On-Kaggle-T4x2-GPU`.
2. Select **Accelerator → GPU T4 x2**.
3. Attach Kaggle Model `dangkhoa2016/openmoss-team-moss-tts-v1-5`.
4. Keep **Internet = ON** for source checkout and dependency bootstrap.
5. By default the notebook uses immutable release tag **`v1.0.0`**. Optionally set **`MOSS_TTS_SOURCE_REF`** to an exact 40-character commit SHA for audit/reproduction. Mutable refs such as `main` are rejected as runtime authority.
6. Use **Restart Session → Run All**. Runtime stages execute in fresh subprocesses after dependency installation, so no additional manual restart is required.

### Tiếng Việt

Trước khi chạy notebook trên Kaggle:

1. Import notebook từ repository `dangkhoa2016/MOSS-TTS-v1.5-On-Kaggle-T4x2-GPU`.
2. Chọn **Accelerator → GPU T4 x2**.
3. Attach Kaggle Model `dangkhoa2016/openmoss-team-moss-tts-v1-5`.
4. Giữ **Internet = ON** để checkout source và bootstrap dependency.
5. Mặc định notebook dùng release tag bất biến **`v1.0.0`**. Có thể đặt **`MOSS_TTS_SOURCE_REF`** bằng SHA commit chính xác 40 ký tự để audit/tái lập. Ref mutable như `main` bị từ chối làm runtime authority.
6. Dùng **Restart Session → Run All**. Các stage runtime chạy trong subprocess mới sau khi cài dependency nên không cần restart thủ công thêm lần nữa.

### Expected evidence / Kết quả cần thấy

The next cell must print `PUBLIC_SOURCE_HEAD=<40-char-sha>` and `PUBLIC_SOURCE_PIN=PASS`. / Cell tiếp theo phải in đúng hai marker này để chứng minh source đã được pin bất biến.

## Step 1 — Immutable public source pin / Ghim mã nguồn bất biến

### English

This cell clones the repository, fetches only the requested immutable source ref, checks out the resolved commit in detached mode, and verifies that an exact commit request resolves to the same SHA. Mutable refs such as `main` fail by design.

### Tiếng Việt

Cell này clone repository, chỉ fetch source ref bất biến được yêu cầu, checkout commit đã resolve ở detached mode và xác minh rằng SHA commit được yêu cầu phải khớp chính xác với SHA thực tế. Ref có thể thay đổi như `main` bị từ chối theo thiết kế.

### Expected evidence / Kết quả cần thấy

- `PUBLIC_SOURCE_REF=<requested-ref>`
- `PUBLIC_SOURCE_HEAD=<40-char-sha>`
- `PUBLIC_SOURCE_PIN=PASS`

In [ ]:
from pathlib import Path
import json, os, re, shutil, subprocess, sys

REPO_URL = "https://github.com/dangkhoa2016/MOSS-TTS-v1.5-On-Kaggle-T4x2-GPU.git"
ROOT = Path("/kaggle/working/moss-tts-v15-public")
REPORTS = ROOT / "results" / "notebook"
SUBPROCESS_ENV = os.environ.copy()
_existing_pythonpath = SUBPROCESS_ENV.get("PYTHONPATH", "")
SUBPROCESS_ENV["PYTHONPATH"] = (
    str(ROOT) if not _existing_pythonpath else str(ROOT) + os.pathsep + _existing_pythonpath
)

# Full SHA-256 verification is intentionally disabled by default because it must
# read and hash the complete ~15.8 GiB BF16 checkpoint. The repository already
# contains canonical digests established during fresh release qualification.
# Set this to True when you want to independently re-verify the attached Kaggle
# checkpoint byte-for-byte against those canonical digests.
#
# Xác minh SHA-256 toàn bộ checkpoint được tắt mặc định vì phải đọc và hash toàn
# bộ checkpoint BF16 ~15.8 GiB. Repository đã có canonical digest được xác nhận
# trong fresh release qualification. Đặt True nếu muốn tự kiểm chứng lại toàn bộ
# checkpoint Kaggle theo từng byte.
VERIFY_CHECKPOINT_SHA256 = False
DEFAULT_SOURCE_REF = "v1.0.0"
SOURCE_REF = os.environ.get("MOSS_TTS_SOURCE_REF", DEFAULT_SOURCE_REF).strip()
COMMIT_SHA = re.compile(r"^[0-9a-f]{40}$")
RELEASE_TAG = re.compile(r"^v\d+\.\d+\.\d+$")

if not (COMMIT_SHA.fullmatch(SOURCE_REF) or RELEASE_TAG.fullmatch(SOURCE_REF)):
    raise SystemExit(
        "MOSS_TTS_SOURCE_REF must be an immutable release tag such as 'v1.0.0' or an exact 40-character commit SHA; "
        "mutable refs such as 'main' are not accepted as runtime authority. Got: " + repr(SOURCE_REF)
    )

if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(ROOT)], check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", SOURCE_REF], cwd=ROOT, check=True)
FETCHED_COMMIT = subprocess.check_output(["git", "rev-parse", "FETCH_HEAD^{commit}"], cwd=ROOT, text=True).strip()
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)

RESOLVED_HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if RESOLVED_HEAD != FETCHED_COMMIT:
    raise SystemExit(f"source pin mismatch: fetched {FETCHED_COMMIT}, checked out {RESOLVED_HEAD}")
if COMMIT_SHA.fullmatch(SOURCE_REF) and RESOLVED_HEAD != SOURCE_REF:
    raise SystemExit(f"source pin mismatch: requested {SOURCE_REF}, checked out {RESOLVED_HEAD}")

SOURCE_PINNED = (
    (COMMIT_SHA.fullmatch(SOURCE_REF) is not None and RESOLVED_HEAD == SOURCE_REF)
    or RELEASE_TAG.fullmatch(SOURCE_REF) is not None
)
assert SOURCE_PINNED, f"source pin could not be proven for {SOURCE_REF} / {RESOLVED_HEAD}"
print(f"PUBLIC_SOURCE_REF={SOURCE_REF}")
print(f"PUBLIC_SOURCE_HEAD={RESOLVED_HEAD}")
print("PUBLIC_SOURCE_PIN=PASS")


## Step 2 — Dependency baseline bootstrap / Cài đặt baseline dependency

### English

This cell installs `requirements/kaggle.txt` from the pinned source revision and verifies every declared version range against the interpreter that will execute the qualification scripts. The production runtime manifest intentionally contains only direct runtime dependencies.

### Tiếng Việt

Cell này cài `requirements/kaggle.txt` từ source revision đã pin và kiểm tra mọi version range đã khai báo bằng đúng interpreter sẽ chạy các qualification script. Runtime manifest dành cho production chỉ giữ các dependency trực tiếp cần thiết.

### Expected evidence / Kết quả cần thấy

- one `DEPENDENCY ... installed=...` line per declared runtime dependency;
- `DEPENDENCY_BASELINE=PASS`;
- `KAGGLE_KERNEL_RESTART_REQUIRED=NO`.

In [ ]:
from importlib import metadata as importlib_metadata
from packaging.requirements import Requirement

REQUIREMENTS = ROOT / "requirements" / "kaggle.txt"
declared = [
    Requirement(line)
    for line in REQUIREMENTS.read_text().splitlines()
    if line.strip() and not line.strip().startswith(("#", "-"))
]
installed_before = {req.name: importlib_metadata.version(req.name) for req in declared}

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS)], check=True)

installed_after = {req.name: importlib_metadata.version(req.name) for req in declared}
violations = [
    f"{req.name}{req.specifier} != {installed_after[req.name]}"
    for req in declared
    if not req.specifier.contains(installed_after[req.name], prereleases=True)
]
changed = sorted(name for name in installed_after if installed_before.get(name) != installed_after[name])

for req in declared:
    print(f"DEPENDENCY {req.name}{req.specifier} installed={installed_after[req.name]}")
assert not violations, "dependency baseline violated: " + ", ".join(violations)
print("DEPENDENCY_BASELINE=PASS")
# Every runtime step below runs in a fresh subprocess started after this install, so the
# upgraded baseline is what actually executes; no extra kernel restart is required.
print("KAGGLE_KERNEL_RESTART_REQUIRED=NO")
if changed:
    print("KAGGLE_KERNEL_RESTART_ADVISED_PACKAGES=" + ",".join(changed))

## Step 3 — GPU T4×2 preflight / Kiểm tra GPU T4×2

### English

Before any model weights are loaded, this cell verifies that the live Kaggle runtime exposes exactly two GPUs and that both normalized device names are `tesla t4`. Any other accelerator topology fails closed.

### Tiếng Việt

Trước khi nạp bất kỳ model weight nào, cell này xác minh Kaggle runtime hiện tại có đúng hai GPU và tên chuẩn hóa của cả hai thiết bị đều là `tesla t4`. Mọi topology accelerator khác đều bị dừng theo nguyên tắc fail-closed.

### Expected evidence / Kết quả cần thấy

- preflight report with `gpu_count = 2`;
- normalized GPU names `tesla t4`, `tesla t4`;
- `GPU_T4X2=PASS`.

In [ ]:
REPORTS.mkdir(parents=True, exist_ok=True)

preflight_run = subprocess.run(
    [sys.executable, "scripts/preflight.py", "--report", str(REPORTS / "preflight.json")],
    cwd=ROOT,
    env=SUBPROCESS_ENV,
    capture_output=True,
    text=True,
)
print(preflight_run.stdout, end="")
if preflight_run.stderr:
    print(preflight_run.stderr, file=sys.stderr, end="")
preflight_run.check_returncode()
preflight_report = json.loads((REPORTS / "preflight.json").read_text())
assert preflight_report["status"] == "PASS"
assert preflight_report["gpu_count"] == 2
assert preflight_report["normalized_gpu_names"] == ["tesla t4", "tesla t4"]
assert "GPU_T4X2=PASS" in preflight_run.stdout
print("PREFLIGHT_REPORT=" + json.dumps(preflight_report, sort_keys=True))
print("GPU_T4X2=PASS")

## Step 4 — Checkpoint identity, dtype and format evidence / Bằng chứng checkpoint, dtype và định dạng

### English

This cell inventories the attached Kaggle Model without materializing the full checkpoint. It always verifies the declared BF16 dtype, safetensors shard structure and tensor headers, and rejects GGUF and quantization. Full-file SHA-256 recomputation is optional because hashing the complete ~15.8 GiB checkpoint can noticeably increase startup time on Kaggle storage. `VERIFY_CHECKPOINT_SHA256` defaults to `False`; set it to `True` when you want this run to independently re-hash every canonical checkpoint file. The canonical digests remain recorded in the repository either way.

### Tiếng Việt

Cell này kiểm kê Kaggle Model đã attach mà không materialize toàn bộ checkpoint. Nó luôn xác minh dtype BF16 đã khai báo, cấu trúc shard safetensors và tensor header, đồng thời từ chối GGUF và quantization. Việc tính lại SHA-256 toàn bộ file là tùy chọn vì hash toàn bộ checkpoint ~15.8 GiB có thể làm tăng đáng kể thời gian khởi động trên Kaggle. `VERIFY_CHECKPOINT_SHA256` mặc định là `False`; đặt thành `True` nếu muốn lần chạy này tự hash lại từng file canonical. Các canonical digest vẫn được lưu trong repository dù bật hay tắt.

### Expected evidence / Kết quả cần thấy

- default demo: `VERIFY_CHECKPOINT_SHA256=False`, `SHA256_RECOMPUTE=SKIPPED`, `SHA256_REFERENCE=PREQUALIFIED`;
- full verification: `VERIFY_CHECKPOINT_SHA256=True`, `SHA256_RECOMPUTE=PASS`, `SHA256_REFERENCE=PASS`;
- `QUANTIZATION=NONE`;
- `GGUF=NO`;
- `MODEL_IDENTITY_AND_SAFETENSORS=PASS`.


In [ ]:
inventory_cmd = [
    sys.executable,
    "scripts/inventory_model.py",
    "--root",
    "/kaggle/input/models",
    "--report",
    str(REPORTS / "inventory.json"),
]
if not VERIFY_CHECKPOINT_SHA256:
    inventory_cmd.append("--skip-sha256")

inventory_run = subprocess.run(
    inventory_cmd,
    cwd=ROOT,
    env=SUBPROCESS_ENV,
    capture_output=True,
    text=True,
)
if inventory_run.stdout:
    print(inventory_run.stdout, end="" if inventory_run.stdout.endswith("\n") else "\n")
if inventory_run.stderr:
    print(inventory_run.stderr, end="" if inventory_run.stderr.endswith("\n") else "\n", file=sys.stderr)
if inventory_run.returncode != 0:
    print(
        "MODEL_ATTACHMENT_HINT: inventory failed. Confirm the Kaggle model "
        "dangkhoa2016/openmoss-team-moss-tts-v1-5 is attached and mounted under /kaggle/input/models.",
        file=sys.stderr,
    )
inventory_run.check_returncode()
inventory_report = json.loads((REPORTS / "inventory.json").read_text())

assert inventory_report["status"] == "PASS"
assert inventory_report["identity"]["declared_dtype"] == "bfloat16"
assert inventory_report["checkpoint_dtype_verification"]["status"] == "PASS"
assert inventory_report["checkpoint_dtype_verification"]["weights_materialized"] is False
assert inventory_report["format"]["gguf"] is False
assert inventory_report["quantization"] == "none"
sha_status = inventory_report["checkpoint_sha256_reference"]["status"]

print(f"VERIFY_CHECKPOINT_SHA256={VERIFY_CHECKPOINT_SHA256}")
if VERIFY_CHECKPOINT_SHA256:
    assert sha_status == "PASS"
    print("SHA256_RECOMPUTE=PASS")
    print("SHA256_REFERENCE=PASS")
else:
    assert sha_status == "SKIPPED"
    print("SHA256_RECOMPUTE=SKIPPED")
    print("SHA256_REFERENCE=PREQUALIFIED")

print("CHECKPOINT_SHARDS=" + str(inventory_report["shard_count"]))
print("QUANTIZATION=" + inventory_report["quantization"].upper())
print("GGUF=NO")
print("MODEL_IDENTITY_AND_SAFETENSORS=PASS")


## Step 5 — Reviewer-facing voice showcase / Showcase giọng nói cho reviewer

### English

The next six samples are presentation-oriented listening cases: two Vietnamese-only, two English-only, and two bidirectional Vietnamese-English code-switch examples. Step 6 adds a seventh IPA pronunciation-control sample. All seven cases run inside **one shared synthesis lifecycle**: the BF16 backbone is loaded once, all token sequences are generated, the backbone is released, then the audio decoder is loaded once and reused for every WAV. This avoids repeatedly reading the ~17 GB checkpoint while preserving the original weights and model architecture.

### Tiếng Việt

Sáu mẫu tiếp theo được thiết kế để reviewer nghe trực tiếp: hai mẫu thuần tiếng Việt, hai mẫu thuần tiếng Anh và hai mẫu code-switch Việt–Anh theo hai chiều. Step 6 bổ sung mẫu thứ bảy để kiểm soát phát âm bằng IPA. Cả bảy mẫu chạy trong **một shared synthesis lifecycle**: backbone BF16 chỉ được nạp một lần, toàn bộ token sequence được generate, backbone được giải phóng, sau đó audio decoder chỉ được nạp một lần và tái sử dụng cho mọi WAV. Cách này tránh đọc lặp checkpoint ~17 GB mà không thay đổi weights hay kiến trúc model.

### Expected evidence / Kết quả cần thấy

The batch runner must report `shared_backbone_load_count=1` and `shared_decoder_load_count=1`. Each sample still renders a playable WAV plus `duration`, `generation_rtf`, `decode_rtf`, and peak VRAM for both GPUs. / Batch runner phải báo `shared_backbone_load_count=1` và `shared_decoder_load_count=1`. Mỗi sample vẫn hiển thị WAV có thể phát cùng `duration`, `generation_rtf`, `decode_rtf` và peak VRAM của cả hai GPU.


In [ ]:
from IPython.display import Audio, display

SHOWCASE_CASES = [
    {"id": "vi_narrative", "language": "Vietnamese", "text": "Buổi sáng ở Nha Trang bắt đầu bằng tiếng sóng nhẹ ngoài cửa sổ. Mình pha một tách cà phê, mở máy tính và bắt đầu một ngày làm việc mới với thật nhiều ý tưởng."},
    {"id": "vi_longform", "language": "Vietnamese", "text": "Mô hình MOSS TTS phiên bản một chấm năm đang chạy checkpoint /biː ɛf sɪkˈstiːn/ nguyên bản trên hai GPU Tesla T4. Hệ thống phân bổ các lớp của mô hình giữa hai GPU, sau đó giải phóng backbone trước khi nạp bộ giải mã âm thanh để kiểm soát bộ nhớ."},
    {"id": "en_narrative", "language": "English", "text": "Early in the morning, the city is quiet and the first light reflects across the water. I open my laptop, review the plan for the day, and begin turning a small idea into something people can actually use."},
    {"id": "en_technical", "language": "English", "text": "This production demo runs the original MOSS TTS version one point five checkpoint in /biː ɛf sɪkˈstiːn/ across two Tesla T4 GPUs. The deployment changes memory placement and execution order, but it does not quantize or modify the model weights."},
    {"id": "mix_vi_en", "language": "Vietnamese", "text": "Xin chào, hôm nay mình đang kiểm tra khả năng chuyển ngôn ngữ của MOSS TTS trên Kaggle. The interesting part is that the sentence can continue naturally in English without starting a separate synthesis job."},
    {"id": "mix_en_vi", "language": "English", "text": "This is an English introduction for the reviewer, but the speaker will switch languages now. Từ thời điểm này, câu nói tiếp tục hoàn toàn bằng tiếng Việt để kiểm tra độ mượt khi chuyển ngôn ngữ theo chiều ngược lại."},
    {"id": "ipa", "language": "English", "text": "Moss TTS is running with /biː ɛf sɪkˈstiːn/ precision on two Tesla T4 GPUs."},
]

cases_path = REPORTS / "showcase-cases.json"
batch_summary_path = REPORTS / "showcase-batch-summary.json"
cases_path.write_text(json.dumps({"cases": SHOWCASE_CASES}, indent=2, ensure_ascii=False))

process = subprocess.Popen(
    [
        sys.executable,
        "scripts/generate_batch.py",
        "--model-root", "/kaggle/input/models",
        "--cases", str(cases_path),
        "--output-dir", str(REPORTS),
        "--summary-report", str(batch_summary_path),
    ],
    cwd=ROOT,
    env=SUBPROCESS_ENV,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
runner_lines = []
assert process.stdout is not None
for line in process.stdout:
    runner_lines.append(line.rstrip())
    print(line, end="", flush=True)
returncode = process.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, process.args)

batch_summary = json.loads(batch_summary_path.read_text())
assert batch_summary["status"] == "PASS"
assert batch_summary["case_count"] == 7
assert batch_summary["shared_backbone_load_count"] == 1
assert batch_summary["shared_decoder_load_count"] == 1

generation_reports = {
    case["id"]: json.loads((REPORTS / f"{case['id']}.json").read_text())
    for case in SHOWCASE_CASES
}

def display_showcase_case(case_id: str):
    """Render one already-generated reviewer sample and its compact evidence."""
    case = next(item for item in SHOWCASE_CASES if item["id"] == case_id)
    report = generation_reports[case_id]
    wav = REPORTS / f"{case_id}.wav"
    assert wav.is_file() and wav.stat().st_size > 0
    print(f"CASE={case_id}")
    print(f"TEXT={case['text']}")
    print(
        "METRICS "
        f"duration={report['audio_duration_seconds']:.2f}s "
        f"generation_rtf={report['generation_rtf']:.3f} "
        f"decode_rtf={report['decode_rtf']:.3f} "
        f"gpu_peak_gib={report['gpu0_peak_alloc_gib']:.3f}/{report['gpu1_peak_alloc_gib']:.3f}"
    )
    display(Audio(str(wav)))
    return report


### Sample 1 — Vietnamese-only natural narration / Lời kể tự nhiên thuần tiếng Việt

### English

This sample evaluates natural Vietnamese narration with everyday vocabulary, punctuation-driven pauses, and connected phrasing across two sentences.

### Tiếng Việt

Mẫu này đánh giá khả năng đọc tiếng Việt tự nhiên với từ vựng đời thường, nhịp nghỉ theo dấu câu và sự liền mạch giữa các cụm từ trong hai câu kể.

### Sample text / Nội dung mẫu

**Language / Ngôn ngữ:** Vietnamese

> Buổi sáng ở Nha Trang bắt đầu bằng tiếng sóng nhẹ ngoài cửa sổ. Mình pha một tách cà phê, mở máy tính và bắt đầu một ngày làm việc mới với thật nhiều ý tưởng.

### What to listen for / Điểm cần nghe

- Vietnamese tones and vowel clarity / dấu thanh và độ rõ nguyên âm;
- punctuation-aware pauses / nhịp nghỉ theo dấu câu;
- continuity across phrases / độ liền mạch giữa các cụm từ.

### Expected evidence / Kết quả cần thấy

A playable `vi_narrative.wav` plus duration, generation/decode RTF and peak VRAM metrics. / Audio `vi_narrative.wav` phát được cùng duration, generation/decode RTF và peak VRAM.

In [ ]:
display_showcase_case("vi_narrative")


### Sample 2 — Vietnamese-only technical speech / Nội dung kỹ thuật thuần tiếng Việt

### English

This longer Vietnamese sample stresses technical terminology, version numbers, GPU names, and multi-clause pacing while keeping the content understandable to a reviewer. Because raw `BF16` was repeatedly mispronounced in Vietnamese, the synthesis input uses the upstream IPA-span pronunciation control `/biː ɛf sɪkˈstiːn/` for that visible term; this changes only the input text, not the model.

### Tiếng Việt

Mẫu tiếng Việt dài hơn này kiểm tra thuật ngữ kỹ thuật, số phiên bản, tên GPU và nhịp đọc của câu nhiều mệnh đề nhưng vẫn phải dễ nghe đối với reviewer. Vì chuỗi `BF16` thô đã nhiều lần bị đọc sai trong tiếng Việt, input synthesis dùng pronunciation control IPA span `/biː ɛf sɪkˈstiːn/` do upstream hỗ trợ cho thuật ngữ hiển thị này; thay đổi chỉ nằm ở input text, không sửa model.

### Sample text / Nội dung mẫu

**Language / Ngôn ngữ:** Vietnamese

> Mô hình MOSS TTS phiên bản một chấm năm đang chạy checkpoint BF16 (`/biː ɛf sɪkˈstiːn/` trong synthesis input) nguyên bản trên hai GPU Tesla T4. Hệ thống phân bổ các lớp của mô hình giữa hai GPU, sau đó giải phóng backbone trước khi nạp bộ giải mã âm thanh để kiểm soát bộ nhớ.

### What to listen for / Điểm cần nghe

- technical terms and version numbers / thuật ngữ kỹ thuật và số phiên bản;
- long-sentence pacing / nhịp đọc câu dài;
- stable Vietnamese intonation / ngữ điệu tiếng Việt ổn định.

### Expected evidence / Kết quả cần thấy

A playable `vi_longform.wav` plus duration, generation/decode RTF and peak VRAM metrics. / Audio `vi_longform.wav` phát được cùng các runtime metric tương ứng.

In [ ]:
display_showcase_case("vi_longform")


### Sample 3 — English-only natural narration / Natural narration thuần tiếng Anh

### English

This sample evaluates natural English prosody using a short narrative with punctuation, connected clauses, and ordinary conversational vocabulary.

### Tiếng Việt

Mẫu này đánh giá prosody tiếng Anh tự nhiên bằng một đoạn kể ngắn có dấu câu, nhiều cụm liên kết và từ vựng hội thoại thông thường.

### Sample text / Nội dung mẫu

**Language / Ngôn ngữ:** English

> Early in the morning, the city is quiet and the first light reflects across the water. I open my laptop, review the plan for the day, and begin turning a small idea into something people can actually use.

### What to listen for / Điểm cần nghe

- English prosody and sentence rhythm / prosody và nhịp câu tiếng Anh;
- punctuation-aware pauses / nhịp nghỉ theo dấu câu;
- natural continuity across a longer narrative / độ tự nhiên xuyên suốt đoạn kể dài hơn.

### Expected evidence / Kết quả cần thấy

A playable `en_narrative.wav` plus duration, generation/decode RTF and peak VRAM metrics. / Audio `en_narrative.wav` phát được cùng các runtime metric tương ứng.

In [ ]:
display_showcase_case("en_narrative")


### Sample 4 — English-only technical speech / Nội dung kỹ thuật thuần tiếng Anh

### English

This sample evaluates technical English narration containing model names, precision terminology, GPU names, and an explicit statement that deployment changes do not alter the checkpoint. The visible term `BF16` is synthesized with the upstream IPA-span pronunciation control `/biː ɛf sɪkˈstiːn/` so the technical reading is deterministic without changing the model.

### Tiếng Việt

Mẫu này đánh giá giọng đọc tiếng Anh kỹ thuật với tên model, thuật ngữ precision, tên GPU và câu mô tả rõ rằng thay đổi cách triển khai không làm thay đổi checkpoint. Thuật ngữ hiển thị `BF16` được synthesis bằng pronunciation control IPA span `/biː ɛf sɪkˈstiːn/` do upstream hỗ trợ để cách đọc kỹ thuật ổn định mà không sửa model.

### Sample text / Nội dung mẫu

**Language / Ngôn ngữ:** English

> This production demo runs the original MOSS TTS version one point five checkpoint in BF16 (`/biː ɛf sɪkˈstiːn/` in the synthesis input) across two Tesla T4 GPUs. The deployment changes memory placement and execution order, but it does not quantize or modify the model weights.

### What to listen for / Điểm cần nghe

- clarity of technical terms / độ rõ của thuật ngữ kỹ thuật;
- numbers and GPU names / số và tên GPU;
- stable emphasis across two sentences / trọng âm ổn định xuyên suốt hai câu.

### Expected evidence / Kết quả cần thấy

A playable `en_technical.wav` plus duration, generation/decode RTF and peak VRAM metrics. / Audio `en_technical.wav` phát được cùng các runtime metric tương ứng.

In [ ]:
display_showcase_case("en_technical")


### Sample 5 — Vietnamese → English code-switch / Code-switch từ tiếng Việt sang tiếng Anh

### English

This sample starts in Vietnamese and switches to English within the same synthesis request. It is designed to reveal whether the transition sounds continuous rather than like two separately generated clips.

### Tiếng Việt

Mẫu này bắt đầu bằng tiếng Việt rồi chuyển sang tiếng Anh trong cùng một synthesis request. Mục tiêu là nghe xem điểm chuyển có liền mạch hay tạo cảm giác như hai clip được ghép lại.

### Sample text / Nội dung mẫu

**Pattern / Hướng chuyển:** Vietnamese → English

> Xin chào, hôm nay mình đang kiểm tra khả năng chuyển ngôn ngữ của MOSS TTS trên Kaggle. The interesting part is that the sentence can continue naturally in English without starting a separate synthesis job.

### What to listen for / Điểm cần nghe

- smooth language transition / độ mượt tại điểm chuyển ngôn ngữ;
- English accent after the switch / accent ở phần tiếng Anh;
- continuity of one synthesis turn / tính liên tục của cùng một lượt synthesis.

### Expected evidence / Kết quả cần thấy

A playable `mix_vi_en.wav` plus duration, generation/decode RTF and peak VRAM metrics. / Audio `mix_vi_en.wav` phát được cùng các runtime metric tương ứng.

In [ ]:
display_showcase_case("mix_vi_en")


### Sample 6 — English → Vietnamese code-switch / Code-switch từ tiếng Anh sang tiếng Việt

### English

This sample exercises the reverse transition: it begins in English and then continues fully in Vietnamese within the same synthesis request.

### Tiếng Việt

Mẫu này kiểm tra chiều chuyển ngược lại: bắt đầu bằng tiếng Anh rồi tiếp tục hoàn toàn bằng tiếng Việt trong cùng một synthesis request.

### Sample text / Nội dung mẫu

**Pattern / Hướng chuyển:** English → Vietnamese

> This is an English introduction for the reviewer, but the speaker will switch languages now. Từ thời điểm này, câu nói tiếp tục hoàn toàn bằng tiếng Việt để kiểm tra độ mượt khi chuyển ngôn ngữ theo chiều ngược lại.

### What to listen for / Điểm cần nghe

- transition from English into Vietnamese / điểm chuyển từ tiếng Anh sang tiếng Việt;
- Vietnamese tones immediately after the switch / dấu thanh ngay sau điểm chuyển;
- consistent rhythm and speaker continuity / nhịp nói và tính liên tục của giọng.

### Expected evidence / Kết quả cần thấy

A playable `mix_en_vi.wav` plus duration, generation/decode RTF and peak VRAM metrics. / Audio `mix_en_vi.wav` phát được cùng các runtime metric tương ứng.

In [ ]:
display_showcase_case("mix_en_vi")


## Step 6 — IPA pronunciation control / Kiểm soát phát âm bằng IPA

### English

Raw acronyms may be spoken differently from what a reviewer expects. This sample demonstrates the upstream-supported **IPA span** feature using `/biː ɛf sɪkˈstiːn/` for “BF16”. It is pronunciation control supplied in the input text, not a model patch.

### Tiếng Việt

Từ viết tắt có thể được model đọc khác với cách reviewer mong đợi. Mẫu này minh họa **IPA span** do upstream hỗ trợ, dùng `/biː ɛf sɪkˈstiːn/` cho “BF16”. Đây là kiểm soát phát âm nằm trong input text, không phải chỉnh sửa model.

### Sample text / Nội dung mẫu

**Language / Ngôn ngữ:** English with an IPA span

> Moss TTS is running with /biː ɛf sɪkˈstiːn/ precision on two Tesla T4 GPUs.

### What to listen for / Điểm cần nghe

Listen specifically to how the IPA span controls “BF16” while the rest of the sentence remains ordinary English. / Tập trung nghe cách IPA điều khiển riêng cụm “BF16” trong khi phần còn lại vẫn là tiếng Anh thông thường.

### Expected evidence / Kết quả cần thấy

A playable `ipa.wav` plus duration, generation/decode RTF and peak VRAM metrics. / Audio `ipa.wav` phát được cùng các runtime metric tương ứng.

In [ ]:
display_showcase_case("ipa")


## Step 7 — Derived scorecard and final machine-readable qualification / Scorecard suy ra và qualification cuối

### English

Every scorecard field is derived from machine-readable reports produced by earlier cells. The final technical gate requires all seven reviewer samples: two Vietnamese-only, two English-only, two code-switch, and one IPA pronunciation-control case. `KAGGLE_PRODUCTION_DEMO=PASS` is printed only after every technical assertion succeeds. Human listening remains separate from the machine gate and should be completed before publicly sharing this post-release run.

### Tiếng Việt

Mọi trường trong scorecard đều được suy ra từ machine-readable report do các cell trước tạo ra. Technical gate cuối yêu cầu đủ bảy mẫu reviewer: hai mẫu thuần tiếng Việt, hai mẫu thuần tiếng Anh, hai mẫu code-switch và một mẫu kiểm soát phát âm IPA. `KAGGLE_PRODUCTION_DEMO=PASS` chỉ được in sau khi toàn bộ assertion kỹ thuật thành công. Human listening vẫn tách biệt với machine gate và cần hoàn tất trước khi chia sẻ công khai post-release run này.

### Expected evidence / Kết quả cần thấy

- scorecard JSON derived from current-run reports;
- `QUANTIZATION=NONE`;
- `GGUF=NO`;
- `MODEL_PATCH=NONE`;
- `HUMAN_LISTENING_REVIEW=PENDING`;
- `KAGGLE_PRODUCTION_DEMO=PASS`;


In [ ]:
all_reports = list(generation_reports.values())
EXPECTED_AUDIO_CASES = {
    "vi_narrative",
    "vi_longform",
    "en_narrative",
    "en_technical",
    "mix_vi_en",
    "mix_en_vi",
    "ipa",
}

scorecard = {
    "public_source_ref": SOURCE_REF,
    "public_source_head": RESOLVED_HEAD,
    "public_source_pin": RESOLVED_HEAD == SOURCE_REF or bool(RELEASE_TAG.match(SOURCE_REF)),
    "gpu_t4x2": preflight_report["status"] == "PASS" and preflight_report["gpu_count"] == 2,
    "model_identity_and_safetensors": inventory_report["status"] == "PASS",
    "declared_dtype_bf16": inventory_report["identity"]["declared_dtype"] == "bfloat16",
    "checkpoint_dtype_verification": inventory_report["checkpoint_dtype_verification"]["status"] == "PASS",
    "verify_checkpoint_sha256": VERIFY_CHECKPOINT_SHA256,
    "checkpoint_sha256_reference": inventory_report["checkpoint_sha256_reference"]["status"],
    "gguf": inventory_report["format"]["gguf"],
    "quantization": inventory_report["quantization"],
    "model_patch": any(report["weight_map_match"]["status"] != "PASS" for report in all_reports),
    "architecture_modified": any(report["weight_map_match"]["status"] != "PASS" for report in all_reports),
    "audio_cases": sorted(generation_reports),
    "generated_steps": {case: report["generated_steps"] for case, report in generation_reports.items()},
    "generation_rtf": {case: report["generation_rtf"] for case, report in generation_reports.items()},
    "decode_rtf": {case: report["decode_rtf"] for case, report in generation_reports.items()},
    "audio_duration_seconds": {case: report["audio_duration_seconds"] for case, report in generation_reports.items()},
    "gpu_peak_alloc_gib": {
        case: [report["gpu0_peak_alloc_gib"], report["gpu1_peak_alloc_gib"]]
        for case, report in generation_reports.items()
    },
    "runtime_versions": all_reports[0]["provenance"]["package_versions"],
    "checkpoint_hashes": inventory_report["files"],
    "human_listening_review": "PENDING",
}

assert scorecard["public_source_pin"] is True
assert scorecard["gpu_t4x2"] is True
assert scorecard["model_identity_and_safetensors"] is True
assert scorecard["declared_dtype_bf16"] is True
assert scorecard["checkpoint_dtype_verification"] is True
assert scorecard["checkpoint_sha256_reference"] == ("PASS" if VERIFY_CHECKPOINT_SHA256 else "SKIPPED")
assert scorecard["gguf"] is False
assert scorecard["quantization"] == "none"
assert scorecard["model_patch"] is False
assert scorecard["architecture_modified"] is False
assert set(scorecard["audio_cases"]) == EXPECTED_AUDIO_CASES
assert all(report["signal"]["status"] == "PASS" for report in all_reports)
assert all(report["model_identity"]["status"] == "PASS" for report in all_reports)

print(json.dumps(scorecard, indent=2, ensure_ascii=False))
print("QUANTIZATION=" + scorecard["quantization"].upper())
print("GGUF=NO")
print("MODEL_PATCH=NONE")
print("HUMAN_LISTENING_REVIEW=PENDING")
print("KAGGLE_PRODUCTION_DEMO=PASS")
